# GCN with PyTorch for Co-Purchase Recommendation

Graph Convolutional Networks turn co-purchase data into a graph and learn item embeddings for recommendation.

[Read the full article](https://jheiduk.com/posts/gcn-pytorch-recommendation/)

In [ ]:
# Deep learning + graph library
!pip install torch torch_geometric

# Numerical utilities
!pip install numpy

## 1. Co-Purchase Graph Construction

We simulate a catalogue of 20 items split into three categories. Items within the same category are co-purchased with probability 0.7; cross-category pairs with 0.05. Nodes are represented by one-hot identity vectors — all graph structure is learned from edges.

In [ ]:
import torch
import numpy as np
from torch_geometric.data import Data

np.random.seed(42)
torch.manual_seed(42)

N_ITEMS = 20  # toy catalogue of 20 items

# Items 0-6: category 0 | 7-13: category 1 | 14-19: category 2
edges = []
for i in range(N_ITEMS):
    for j in range(i + 1, N_ITEMS):
        same_cat = (i // 7) == (j // 7)
        if np.random.random() < (0.7 if same_cat else 0.05):
            edges.extend([[i, j], [j, i]])  # undirected

edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
x = torch.eye(N_ITEMS, dtype=torch.float)  # one-hot node features

data = Data(x=x, edge_index=edge_index, num_nodes=N_ITEMS)
print(f"Nodes: {data.num_nodes} | Edges: {data.num_edges}")

## 2. GCN Encoder

A two-layer GCN maps the N×N identity matrix to N×32 embeddings. `GCNConv` implements the symmetrically normalised message passing rule from Kipf & Welling (2017).

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv

class GCNEncoder(nn.Module):
    def __init__(self, in_channels: int, hidden: int, out_channels: int):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden)
        self.conv2 = GCNConv(hidden, out_channels)

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=0.3, training=self.training)  # regularise
        return self.conv2(x, edge_index)

model = GCNEncoder(in_channels=N_ITEMS, hidden=64, out_channels=32)
print(model)

## 3. Training with Link Prediction

The model is trained to score real co-purchase pairs higher than random pairs, using binary cross-entropy on dot-product scores. `negative_sampling` draws item pairs that share no edge, in a 1:1 ratio with positive edges.

In [ ]:
from torch_geometric.utils import negative_sampling

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

def bce_link_loss(z, pos_edge_index, neg_edge_index):
    pos_score = (z[pos_edge_index[0]] * z[pos_edge_index[1]]).sum(-1)
    neg_score = (z[neg_edge_index[0]] * z[neg_edge_index[1]]).sum(-1)
    scores = torch.cat([pos_score, neg_score])
    labels = torch.cat([
        torch.ones(pos_score.size(0)),
        torch.zeros(neg_score.size(0)),
    ])
    return F.binary_cross_entropy_with_logits(scores, labels)

model.train()
for epoch in range(1, 301):
    optimizer.zero_grad()
    z = model(data.x, data.edge_index)
    neg_edge = negative_sampling(
        edge_index=data.edge_index,
        num_nodes=N_ITEMS,
        num_neg_samples=data.edge_index.size(1),  # 1:1 positive/negative ratio
    )
    loss = bce_link_loss(z, data.edge_index, neg_edge)
    loss.backward()
    optimizer.step()
    if epoch % 100 == 0:
        print(f"Epoch {epoch:3d} | Loss: {loss.item():.4f}")

## 4. Generating Recommendations

After training, embeddings are L2-normalised so that cosine similarity equals the dot product. For each anchor item, we mask out the self-score and known co-purchase neighbours, then return the top-k highest-scoring items.

In [ ]:
model.eval()
with torch.no_grad():
    embeddings = model(data.x, data.edge_index)
    embeddings = F.normalize(embeddings, p=2, dim=-1)  # unit vectors → cosine = dot product

def recommend(item_id: int, embeddings, top_k: int = 5) -> list:
    scores = embeddings @ embeddings[item_id]    # cosine similarity to all items
    scores[item_id] = -1                          # exclude self
    known = data.edge_index[1][data.edge_index[0] == item_id]
    scores[known] = -1                            # exclude known co-purchases
    return scores.topk(top_k).indices.tolist()

# Anchors 0, 7, 14 are the category representatives
# Correct recs should stay within the same category (0-6, 7-13, 14-19)
for anchor in [0, 7, 14]:
    recs = recommend(anchor, embeddings)
    print(f"Item {anchor:2d} → top-5 recommendations: {recs}")